# 08. 인터럽트와 타임 트래블 — 실행 중단, 승인, 되감기

## 학습 목표

`interrupt()`로 실행을 중단하고, `Command(resume=...)`로 재개합니다. 타임 트래블로 이전 상태로 돌아갑니다.

- Human-in-the-loop 패턴을 구현할 수 있습니다
- Functional API에서도 interrupt를 사용할 수 있습니다
- 체크포인트 히스토리를 활용한 타임 트래블을 수행할 수 있습니다
- `update_state()`로 외부에서 상태를 수정할 수 있습니다

## 8.1 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")
print("모델 준비 완료")

모델 준비 완료


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 8.2 interrupt() — 실행을 중단하고 사람의 입력을 기다립니다

- `interrupt(value)`: 현재 상태를 체크포인트에 저장하고 실행을 중단합니다
- `Command(resume=value)`: 중단된 지점에서 값을 전달하며 재개합니다

이 패턴은 민감한 작업 전에 사람의 승인을 받거나, 추가 정보를 입력받을 때 사용합니다.

In [3]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict


class ReviewState(TypedDict):
    document: str
    approved: bool
    final_result: str


def draft_document(state: ReviewState) -> dict:
    return {
        "document": f"초안: {state.get('document', '주제')}에 대한 중요 문서"
    }


def human_review(state: ReviewState) -> dict:
    # 사람의 승인을 기다립니다
    decision = interrupt(
        {
            "document": state["document"],
            "question": "이 문서를 승인하시겠습니까? (yes/no)"
        }
    )

    return {
        "approved": decision == "yes"
    }


def finalize(state: ReviewState) -> dict:
    if state["approved"]:
        return {
            "final_result": f"승인됨: {state['document']}"
        }

    return {
        "final_result": f"거절됨: {state['document']}"
    }


builder = StateGraph(ReviewState)

builder.add_node("draft", draft_document)
builder.add_node("review", human_review)
builder.add_node("finalize", finalize)

builder.add_edge(START, "draft")
builder.add_edge("draft", "review")
builder.add_edge("review", "finalize")
builder.add_edge("finalize", END)

graph = builder.compile(
    checkpointer=InMemorySaver()
)

config = {
    "configurable": {
        "thread_id": "review-1"
    }
}

# Step 1: 실행 → review 노드에서 중단
result = graph.invoke(
    {
        "document": "AI 안전성"
    },
    {**config, **lf_config}
)

print("Step 1 - review에서 중단됨")

state = graph.get_state(config)

print(f"  다음 노드: {state.next}")
print(f"  인터럽트 값: {state.tasks}")

Step 1 - review에서 중단됨
  다음 노드: ('review',)
  인터럽트 값: (PregelTask(id='478b41b2-f068-6fb7-cc7f-e9a49c46793d', name='review', path=('__pregel_pull', 'review'), error=None, interrupts=(Interrupt(value={'document': '초안: AI 안전성에 대한 중요 문서', 'question': '이 문서를 승인하시겠습니까? (yes/no)'}, id='f0c89de1e19ac278a95201cae2f56848'),), state=None, result=None),)


## 8.3 Command(resume=...) — 중단된 실행을 재개합니다

`Command(resume=value)`를 사용하면 `interrupt()`가 호출된 지점에서 실행이 재개됩니다. `resume`에 전달한 값이 `interrupt()`의 반환값이 됩니다.

In [4]:
# Step 2: 승인하여 재개

result = graph.invoke(
    Command(resume="yes"),
    {**config, **lf_config}
)

print(f"Step 2 - 승인으로 재개됨")
print(f"  결과: {result['final_result']}")

Step 2 - 승인으로 재개됨
  결과: 승인됨: 초안: AI 안전성에 대한 중요 문서


## 8.4 Functional API에서의 interrupt

Functional API(`@entrypoint`, `@task`)에서도 동일하게 `interrupt()`를 사용할 수 있습니다.

In [5]:
from langgraph.func import entrypoint, task

@task
def generate_proposal(topic: str) -> str:
    response = model.invoke(f"다음에 대한 한 문장 제안을 작성해주세요: {topic}", config=lf_config)
    return response.content

@entrypoint(checkpointer=InMemorySaver())
def proposal_workflow(topic: str) -> dict:
    proposal = generate_proposal(topic).result()

    # 사람의 승인 대기
    approval = interrupt({
        "proposal": proposal,
        "action": "승인 또는 거절하세요"
    })

    return {
        "proposal": proposal,
        "approved": approval,
    }

config = {"configurable": {"thread_id": "proposal-1"}}

# 실행 → 중단
result = proposal_workflow.invoke("재생 에너지", {**config, **lf_config})
print("제안 워크플로 중단됨")

# 재개
result = proposal_workflow.invoke(Command(resume="승인"), {**config, **lf_config})
print(f"최종: {result}")

제안 워크플로 중단됨
최종: {'proposal': '재생 에너지는 자연에서 반복적으로 얻을 수 있어 환경 오염을 줄이는 친환경 에너지원입니다.', 'approved': '승인'}


## 8.5 타임 트래블 — 이전 체크포인트로 되돌아가기

LangGraph의 체크포인트 시스템은 모든 실행 상태를 저장합니다. `get_state_history()`로 이전 체크포인트를 조회하고, 특정 시점으로 되돌아갈 수 있습니다.

In [6]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain.messages import HumanMessage

def chatbot(state: MessagesState) -> dict:
    response = model.invoke(state["messages"], config=lf_config)
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "timetravel-1"}}

# 3번의 대화
graph.invoke({"messages": [HumanMessage(content="제가 좋아하는 색은 파랑입니다.")]}, {**config, **lf_config})
graph.invoke({"messages": [HumanMessage(content="제가 좋아하는 음식은 피자입니다.")]}, {**config, **lf_config})
graph.invoke({"messages": [HumanMessage(content="저는 서울에 살고 있습니다.")]}, {**config, **lf_config})

# 히스토리에서 특정 체크포인트 선택
history = list(graph.get_state_history(config))
print(f"전체 체크포인트 수: {len(history)}")

# 두 번째 대화 시점으로 되돌아가기
target = history[2]  # older checkpoint
target_config = target.config
print(f"\n체크포인트에서 재생: {target_config['configurable']['checkpoint_id'][:20]}...")

# 그 시점에서 새로운 대화 시작
result = graph.invoke(
    {"messages": [HumanMessage(content="제가 좋아하는 음식은 뭔가요?")]},
    {**target_config, **lf_config},
)
print(f"응답: {result['messages'][-1].content[:200]}")

전체 체크포인트 수: 9

체크포인트에서 재생: 1f1196e4-19d1-6858-8...


응답: 당신이 좋아하는 음식은 **피자**입니다! 아까 “제가 좋아하는 음식은 피자입니다.”라고 말씀해 주셨거든요. 피자를 정말 좋아하시는 것 같네요!


## 8.6 update_state() — 타임 트래블 + 상태 수정

`update_state()`를 사용하면 외부에서 그래프의 상태를 직접 수정할 수 있습니다. 이는 디버깅, 테스트, 또는 수동 개입이 필요한 경우에 유용합니다.

In [7]:
# 현재 상태에 정보 추가
from langchain.messages import AIMessage

graph.update_state(config, {
    "messages": [AIMessage(content="(업데이트: 사용자는 고양이도 좋아합니다.)")]
})

result = graph.invoke(
    {"messages": [HumanMessage(content="제 선호에 대해 뭘 알고 있나요?")]},
    {**config, **lf_config}
)
print("상태 업데이트 후:", result["messages"][-1].content[:200])

상태 업데이트 후: 지금까지 말씀해 주신 내용을 바탕으로, 당신의 선호에 대해 다음과 같이 알고 있습니다:

1. **좋아하는 색:** 파랑  
2. **좋아하는 음식:** 피자

혹시 더 공유하고 싶으신 취향이나 선호가 있으시면 알려주세요! 앞으로 대화에서 참고해서 더 맞춤형으로 답변드릴 수 있습니다. 😊


## 8.7 요약

이 노트북에서 학습한 핵심 기능을 정리합니다.

| 기능 | API | 설명 |
|------|-----|------|
| `interrupt(value)` | 양쪽 | 실행 중단, 값 전달 |
| `Command(resume=value)` | 양쪽 | 중단 지점에서 재개 |
| `get_state_history()` | Graph | 체크포인트 이력 조회 |
| `update_state()` | Graph | 외부에서 상태 수정 |

**interrupt와 타임 트래블**은 프로덕션 AI 애플리케이션에서 핵심적인 기능입니다:
- **interrupt**: 민감한 작업 전 사람의 승인을 받을 수 있습니다
- **타임 트래블**: 이전 상태로 되돌아가 다른 경로를 탐색할 수 있습니다
- **update_state**: 외부에서 상태를 수정하여 실행 흐름을 조정할 수 있습니다

### 다음 단계
→ **[09_subgraphs.ipynb](./09_subgraphs.ipynb)**: 서브그래프를 배웁니다.
